# cursor

> Cursor's models via the `cursor-agent` CLI or SDK. Not a completion endpoint but an agent with its own prompt overhead, around 16k input tokens a turn.

Defaults: `mode='ask'`, sandbox on, `shell` disallowed. `CursorChat.local` is `False`, so treat
prompts as you would prompts you send to Cursor.


In [ ]:
#| default_exp cursor

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json, os, shutil, subprocess, warnings
from fastcore.all import Path, store_attr, patch, ifnone
from rishi import core
from rishi.core import *

In [ ]:
#| export
_all_ = ['UsageStats', 'ChatCallback', 'run_cbs', 'resp_text', 'thought', 'Resp', 'StreamFormatter',
         'display_stream', 'truncated', 'hitl_policy', 'extract_fence', 'mk_toolspec', 'ToolCall']

In [ ]:
from fastcore.test import test_eq, test_fail

## The wire

One turn is one `cursor-agent` process. There is no system-prompt channel and no message list, so the
whole conversation is rendered into the single prompt the CLI takes. The other re-sending backends
(llama, remote) do the same, which is what lets eviction and `reconfigure` behave here exactly as
they do everywhere else. The alternative, `--resume <session_id>`, would keep Cursor's own
server-side session and let it drift from `chat.hist` the moment anything edited the history.

Tools go out as tags. The CLI hands back text, never a structured call, so the schemas go into the
prompt through `tag_tools_sp` and the calls come back out of the reply through `parse_tool_tags`.
That is the protocol `RemoteChat(tool_mode='tags')` uses for a transport whose tool channel is shut.

The ids themselves come from Cursor, the only thing that knows what a given account can reach.
`cursor_models()` asks it, through `Cursor.models.list` on the SDK path and `cursor-agent models` on
the CLI one. `CURSOR_MODELS` names them for the ergonomics, but adding an id to that dict does not
make Cursor accept it, and Cursor changes the list without asking.

The plain names work on both paths. `cursor-agent -p --model grok-4.5` is accepted even though
`cursor-agent models` lists `cursor-grok-4.5-high` and friends. What differs is how you ask for an
effort level: a parameter on the SDK (`ModelParameterValue(id='fast', value='true')`), and a
decorated name or the bracket form `'claude-opus-4-8[context=1m,effort=high,fast=false]'` on the CLI.
rishi passes the id through verbatim either way, so an id Cursor does not know fails at the far end
rather than here. That is what `cursor_models()` is for.

Going through `Chat` needs the `cursor/` prefix. `claude-opus-5` and `grok-4.5` are also hosted-API
names, so `Chat('grok-4.5')` is `rishi.remote`'s. `CursorChat(grok45)` has nothing to infer.

In [ ]:
#| export
CURSOR_BIN = 'cursor-agent'   #: the CLI rishi drives; override per chat with `bin=`

# Cursor's own ids, as the SDK reports them - and as the CLI accepts them too, though `cursor-agent
# models` lists decorated variants instead (`cursor-grok-4.5-high`, `-low-fast`) with the effort baked
# into the name. The SDK takes the effort as a model parameter, so the plain name is the one that
# works everywhere. `cursor_models()` reports what the active path lists, and is the authority.
cursor_default = 'default'
grok45         = 'grok-4.5'
composer25     = 'composer-2.5'
opus5          = 'claude-opus-5'
opus48         = 'claude-opus-4-8'
opus47         = 'claude-opus-4-7'
opus46         = 'claude-opus-4-6'
opus45         = 'claude-opus-4-5'
fable5         = 'claude-fable-5'
sonnet5        = 'claude-sonnet-5'
sonnet46       = 'claude-sonnet-4-6'
sonnet45       = 'claude-sonnet-4-5'
sonnet4        = 'claude-sonnet-4'
haiku45        = 'claude-haiku-4-5'
gpt56_sol      = 'gpt-5.6-sol'
gpt56_terra    = 'gpt-5.6-terra'
gpt56_luna     = 'gpt-5.6-luna'
gpt55          = 'gpt-5.5'
gpt54          = 'gpt-5.4'
gpt54_mini     = 'gpt-5.4-mini'
gpt54_nano     = 'gpt-5.4-nano'
gpt53_codex    = 'gpt-5.3-codex'
gpt52          = 'gpt-5.2'
gpt51          = 'gpt-5.1'
gpt5_mini      = 'gpt-5-mini'
gemini36_flash = 'gemini-3.6-flash'
gemini35_flash = 'gemini-3.5-flash'
gemini31_pro   = 'gemini-3.1-pro'
gemini3_flash  = 'gemini-3-flash'
gemini25_flash = 'gemini-2.5-flash'
kimi_k3        = 'kimi-k3'
kimi_k27_code  = 'kimi-k2.7-code'
glm52          = 'glm-5.2'

#: Every id above, for anything that wants to offer the list rather than reach for one of them.
CURSOR_MODELS = {'cursor_default': cursor_default, 'grok45': grok45, 'composer25': composer25,
                 'opus5': opus5, 'opus48': opus48, 'opus47': opus47, 'opus46': opus46, 'opus45': opus45,
                 'fable5': fable5, 'sonnet5': sonnet5, 'sonnet46': sonnet46, 'sonnet45': sonnet45,
                 'sonnet4': sonnet4, 'haiku45': haiku45, 'gpt56_sol': gpt56_sol, 'gpt56_terra': gpt56_terra,
                 'gpt56_luna': gpt56_luna, 'gpt55': gpt55, 'gpt54': gpt54, 'gpt54_mini': gpt54_mini,
                 'gpt54_nano': gpt54_nano, 'gpt53_codex': gpt53_codex, 'gpt52': gpt52, 'gpt51': gpt51,
                 'gpt5_mini': gpt5_mini, 'gemini36_flash': gemini36_flash, 'gemini35_flash': gemini35_flash,
                 'gemini31_pro': gemini31_pro, 'gemini3_flash': gemini3_flash, 'gemini25_flash': gemini25_flash,
                 'kimi_k3': kimi_k3, 'kimi_k27_code': kimi_k27_code, 'glm52': glm52}

def cursor_bin(bin=CURSOR_BIN):
    "Absolute path to the `cursor-agent` binary, or a `FileNotFoundError` that says how to get one."
    if (p := shutil.which(bin)): return p
    raise FileNotFoundError(
        f'{bin!r} is not on $PATH. Install the Cursor CLI (https://cursor.com/cli) and run '
        f'`{bin} login`; rishi drives it as a subprocess and never reads your credentials.')

def cursor_via(via=None, api_key=None):
    "Which path to take: what you named, else the SDK when there is a key and a package for it."
    if via not in (None, 'sdk', 'cli'): raise ValueError(f"via must be 'sdk', 'cli' or None, not {via!r}")
    return via or ('sdk' if sdk_available(api_key) else 'cli')

def sdk_available(api_key=None):
    "Is the Python SDK usable here: installed, with a key to use it with?"
    if not (api_key or os.getenv('CURSOR_API_KEY')): return False
    try:
        from cursor_sdk import Cursor # noqa: F401
        return True
    except ImportError: return False

def cursor_models(bin=CURSOR_BIN, api_key=None, via=None):
    """The model ids this account can reach. Cursor is the source of truth, not a table in here.

    Asked through whichever credential there is: `Cursor.models.list` when the SDK has a key, and
    `cursor-agent models` otherwise. The SDK path has no business shelling out to a CLI it does not
    otherwise need, and a machine with a key but no CLI installed is an ordinary machine.
    """
    if cursor_via(via, api_key) == 'sdk':
        from cursor_sdk import Cursor
        return [m.id for m in Cursor.models.list(api_key=api_key)]
    out = subprocess.run([cursor_bin(bin), 'models'], capture_output=True, text=True).stdout
    return [l.split(' - ')[0].strip() for l in out.splitlines() if ' - ' in l]

#: rishi asks for read-only by default, and the two paths spell that differently: the CLI has `ask`
#: (Q&A) and `plan`, the SDK has only `agent` and `plan`. `ask` therefore means `plan` to the SDK -
#: passing it through would serialize to an enum Cursor does not know and be ignored, which is the
#: worst of the three outcomes: a read-only request that quietly runs an agent that can write.
_sdk_modes = {'ask': 'plan', 'plan': 'plan', 'agent': 'agent'}

def sdk_mode(mode):
    "rishi's mode in the SDK's vocabulary, or a `ValueError` rather than a silently ignored one."
    if mode is None: return None
    if mode not in _sdk_modes: raise ValueError(f'unknown mode {mode!r}; use {", ".join(_sdk_modes)} or None')
    return _sdk_modes[mode]

def cli_mode(mode):
    "rishi's mode in the CLI's vocabulary. `--mode` takes `ask` and `plan` only, and the agent is the default."
    if mode in (None, 'agent'): return None
    if mode not in _sdk_modes: raise ValueError(f'unknown mode {mode!r}; use {", ".join(_sdk_modes)} or None')
    return mode

def cursor_model(model, effort=None, fast=None, via='cli'):
    """A model id with its effort and speed attached, spelled the way the active path spells them.

    The SDK carries both as model *parameters* on a `ModelSelection`. The CLI takes the bracket form
    (`grok-4.5[effort=high,fast=true]`) it documents. Same two arguments either way, so a chat that
    moves between paths asks for the same thing rather than being rewritten.
    """
    if not effort and fast is None: return model
    vals = ([('effort', str(effort))] if effort else []) + ([('fast', str(bool(fast)).lower())] if fast is not None else [])
    if via != 'sdk': return f"{model}[{','.join(f'{k}={v}' for k, v in vals)}]"
    from cursor_sdk import ModelSelection, ModelParameterValue
    return ModelSelection(id=model, params=[ModelParameterValue(id=k, value=v) for k, v in vals])

def norm_cursor(d, model=None):
    "A `cursor-agent` JSON result -> a rishi `Resp`, with `<tool_call>` tags read out of the text."
    if d.get('is_error'): raise RuntimeError(f"cursor-agent failed: {d.get('result') or d.get('subtype')}")
    text, th = split_think(d.get('result') or '')
    text, tcs = parse_tool_tags(text)
    res = {'role': 'assistant', 'content': text}
    if th: res['channels'] = {'thought': th}
    if tcs: res['tool_calls'] = tcs
    res['usage'] = norm_cursor_usage(d.get('usage'), model)
    return Resp(res)

def norm_cursor_usage(u, model=None):
    "`cursor-agent`'s usage block -> rishi's, so a Cursor turn adds up with a local one."
    if not u: return {}
    pt, ct = u.get('inputTokens', 0), u.get('outputTokens', 0)
    return {'prompt_tokens': pt, 'completion_tokens': ct, 'total_tokens': pt + ct,
            'cached_tokens': u.get('cacheReadTokens', 0), 'model': model}

In [ ]:
# the transcript keeps roles and tool calls, so a re-sent conversation reads as one
hist = [{'role': 'user', 'content': 'add 1 and 2'},
        {'role': 'assistant', 'content': 'on it', 'tool_calls': [ToolCall('add', {'a': 1, 'b': 2})]},
        {'role': 'tool', 'name': 'add', 'content': '3'}]
p = render_prompt(hist, sp='Be terse.')
assert p.startswith('Be terse.\n\n## User\nadd 1 and 2')
assert '<tool_call>' in p and '"name": "add"' in p and '## Tool result (add)\n3' in p
test_eq(render_prompt([], sp='Be terse.'), 'Be terse.')

# a reply is a `Resp` like any other, tags parsed out and usage folded into rishi's shape
r = norm_cursor({'result': 'on it\n<tool_call>{"name": "add", "arguments": {"a": 1}}</tool_call>',
                 'usage': {'inputTokens': 16391, 'outputTokens': 30, 'cacheReadTokens': 896}}, 'grok')
test_eq(resp_text(r), 'on it')
test_eq(r['tool_calls'][0]['function'], {'name': 'add', 'arguments': {'a': 1}})
test_eq(r['usage'], {'prompt_tokens': 16391, 'completion_tokens': 30, 'total_tokens': 16421,
                     'cached_tokens': 896, 'model': 'grok'})
test_fail(lambda: norm_cursor({'is_error': True, 'result': 'not logged in'}), contains='not logged in')


# the named ids are the SDK's: a base model, no `cursor-` prefix and no effort baked into the name
test_eq((grok45, opus5, cursor_default), ('grok-4.5', 'claude-opus-5', 'default'))
assert not any(i.startswith('cursor-') for i in CURSOR_MODELS.values())
assert not any(i.endswith(('-high', '-low', '-medium', '-xhigh', '-max', '-fast')) for i in CURSOR_MODELS.values())
assert len(set(CURSOR_MODELS.values())) == len(CURSOR_MODELS)      # no id named twice

# ...and they route to this backend whichever family they name, prefix or not
for nm in ('grok45', 'opus5', 'gemini31_pro', 'kimi_k3', 'composer25'):
    test_eq(split_runtime(f'cursor/{CURSOR_MODELS[nm]}')[0], 'cursor')
test_eq(infer_runtime('cursor-grok-4.5-high'), 'cursor')           # the CLI dialect still infers


# discovery follows the credential: the SDK when there is a key for it, the CLI otherwise
import cursor_sdk, types
_real_models = cursor_sdk.Cursor.models
try:
    cursor_sdk.Cursor.models = types.SimpleNamespace(
        list=lambda **kw: [types.SimpleNamespace(id='cursor-grok-4.5-high'), types.SimpleNamespace(id='auto')])
    test_eq(cursor_models(via='sdk'), ['cursor-grok-4.5-high', 'auto'])
finally: cursor_sdk.Cursor.models = _real_models

_real_run, _real_bin = subprocess.run, cursor_bin
try:   # the CLI branch parses `id - Label` lines and ignores the tip at the bottom
    cursor_bin = lambda bin=CURSOR_BIN: '/fake/cursor-agent'
    subprocess.run = lambda *a, **kw: types.SimpleNamespace(
        stdout='Available models\n\nauto - Auto (default)\ncursor-grok-4.5-low - Cursor Grok 4.5 Low\n\nTip: use --model\n')
    test_eq(cursor_models(via='cli'), ['auto', 'cursor-grok-4.5-low'])
finally: subprocess.run, cursor_bin = _real_run, _real_bin

### Cursor's own tools, carrying rishi's

`cursor-agent` takes tool definitions of its own, `custom_tools` on the SDK, and calls them like any
other tool. That is a real channel, so rishi uses it rather than asking a model to punctuate
`<tool_call>` tags in prose. `CursorToolHandler` hands Cursor the schemas and runs each call back
through the same approval, budget and history the tag loop uses. `tool_channel` says which one a chat
is on.

In [ ]:
#| export
class CursorToolHandler:
    "Bridge Cursor's custom tools to Chat callbacks, HITL approval, the tool-call budget, and history."
    def __init__(self, chat): self.chat = chat

    def custom_tools(self):
        "rishi's tools as Cursor `CustomTool`s, or `None` when this chat is on the tags channel."
        from cursor_sdk import CustomTool
        if self.chat.tool_channel != 'native': return None
        return {(fn := ts['function'])['name']: CustomTool(execute=self._exec(fn['name']),
                    description=fn.get('description', ''), input_schema=fn.get('parameters') or {})
                for ts in self.chat.toolspecs}

    def _exec(self, name):
        "The callback Cursor runs for `name`: approve, run, record, hand the result back."
        def call(args, ctx=None):
            c = self.chat
            tc = ToolCall(name, dict(args or {}))
            ok, denial = c._approve1(tc)
            c.hist.append({'role': 'assistant', 'content': '', 'tool_calls': [tc]})
            out = c.call_tool(tc) if ok else denial
            c._record_tool(tc, out)
            return str(out)
        return call

## CursorChat

Conservative defaults, as above. Use plain ids from `rishi.cursor` (`grok45`, `composer25`, ...) or
`Chat('cursor/grok-4.5')` so routing does not hit the hosted API.


In [ ]:
#| export
class CursorChat(ToolLoopMixin, Chat):
    "Chat against a Cursor CLI model, with the same `rishi.core.Chat` API, driving `cursor-agent` headless."
    _runtime = 'cursor'
    _dflt_cbs = [UsageCallback, ToolReminderCallback, SlidingWindowCallback]
    mk_content, mk_msg, mk_msgs = staticmethod(mk_oai_content), staticmethod(mk_oai_msg), staticmethod(mk_oai_msgs)
    local = False   #: the binary is local, the model is not - see the note at the top of this page

    def __init__(self, model=None, *, runtime=None, model_path=None, sp='', messages=None, tools=None,
                 ctx_limit=None, approve=None, tool_max_len=None, max_steps=10, parallel_tools=False,
                 max_parallel_tools=None, final_prompt=dflt_final_prompt_,
                 mode='ask',          # 'ask' (read-only Q&A) or 'plan'; None lets cursor-agent edit and run things
                 sandbox='enabled',   # 'enabled'/'disabled'; None leaves the CLI's own config alone
                 trust=False,         # trust this workspace without prompting - the CLI refuses an untrusted one
                 workspace=None,      # directory cursor-agent works in; None -> the cwd
                 effort=None,         # 'low'/'medium'/'high'/'xhigh'/'max'; None -> the model's own default
                 fast=None,           # ask for the fast build of the model; None -> the model's own default
                 via=None,            # 'sdk' or 'cli'; None -> the SDK when there is a key and the package
                 api_key=None,        # SDK key; None -> $CURSOR_API_KEY. The CLI path needs none of this
                 cursor_tools=None,        # Cursor's *own* tools, allowlisted; the SDK path only
                 cursor_disallowed=('shell',),  # ...and the ones it may never use; the SDK path only
                 bin=CURSOR_BIN, timeout=600, cbs=None, default_cbs=True):
        self.model_id = core.split_runtime(model)[1] or grok45
        self._set_tools(tools)
        self.tool_handler = CursorToolHandler(self)
        store_attr('mode,sandbox,trust,workspace,bin,timeout,api_key,cursor_tools,cursor_disallowed,effort,fast')
        self.via = cursor_via(via, api_key)
        if not self.use_sdk and cli_mode(self.mode) is None and (cursor_tools is not None or cursor_disallowed):
            warnings.warn('the cursor-agent CLI has no tool allowlist: cursor_tools/cursor_disallowed are '
                          "the SDK's. On this path `mode` and `sandbox` are what hold the agent back.")
        self._agent, self._sent = None, 0
        self.ctx_limit, self._ctx_tokens = ctx_limit, 0
        self._setup(model=model, sp=sp, messages=messages, tools=tools, approve=approve,
                    tool_max_len=tool_max_len, max_steps=max_steps, parallel_tools=parallel_tools,
                    max_parallel_tools=max_parallel_tools, final_prompt=final_prompt, cbs=cbs, default_cbs=default_cbs)

    @property
    def tool_channel(self):
        "Where this chat's tool schemas travel: Cursor's own custom tools, or tags in the prompt."
        return 'native' if self.use_sdk and self.toolspecs and sdk_mode(self.mode) == 'agent' else 'tags'

    @property
    def use_sdk(self):
        "Is this chat going through the Python SDK rather than the CLI?"
        return self.via == 'sdk'

    @property
    def token_count(self):
        "the prompt this chat would send next.cursor returns 0 so, we estimate"
        return est_tokens(self._prompt())

    def _cmd(self, fmt):
        "The `cursor-agent` command line for one turn, minus the prompt."
        cmd = [cursor_bin(self.bin), '-p', '--output-format', fmt,
               '--model', cursor_model(self.model_id, self.effort, self.fast)]
        if (m := cli_mode(self.mode)): cmd += ['--mode', m]
        if self.sandbox: cmd += ['--sandbox', self.sandbox]
        if self.trust: cmd += ['--trust']
        if self.workspace: cmd += ['--workspace', str(self.workspace)]
        return cmd

    def _sp(self):
        "The briefing, plus the tool schemas when this chat has no native channel to carry them."
        return self.sp if self.tool_channel == 'native' else tag_tools_sp(self.toolspecs, self.sp)

    def _prompt(self):
        "This turn's whole conversation as text, with the tool schemas in it when there are any."
        return render_prompt(self.hist, self._sp())

    def _run(self, fmt, prompt=None):
        "Run one turn and return the finished process. A non-zero exit is the CLI's message, not a traceback."
        r = subprocess.run(self._cmd(fmt) + [ifnone(prompt, self._prompt())],
                           capture_output=True, text=True, timeout=self.timeout)
        if r.returncode != 0: raise RuntimeError(f'cursor-agent exited {r.returncode}: {(r.stderr or r.stdout).strip()[:400]}')
        return r

    def _note_usage(self, r):
        "Remember what the turn cost. This is billing volume, not occupancy. See `token_count`."
        self._ctx_tokens = (r.get('usage') or {}).get('total_tokens') or self._ctx_tokens
        return r

    def _recreate_conv(self):
        "rishi's history moved under eviction or `reconfigure`, so the live agent's memory of it is wrong."
        self.close()

    def _model_step(self, max_output_tokens=None):
        "One wire call: through the live agent when there is one, else a whole conversation through the CLI."
        if self.use_sdk: return self._sdk_step(max_output_tokens)
        return self._note_usage(norm_cursor(json.loads(self._run('json').stdout), self.model_id))

    def _stream_step(self, max_output_tokens=None):
        "The same turn as `stream-json`: thinking deltas on their own channel, text as it lands."
        if self.use_sdk:
            yield from self._sdk_stream_step(max_output_tokens)
            return
        proc = subprocess.Popen(self._cmd('stream-json') + ['--stream-partial-output', self._prompt()],
                                stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        split, res = StreamSplit(), None
        with killed_on_exit(proc):
            for line in proc.stdout:
                if not (line := line.strip()): continue
                try: o = json.loads(line)
                except json.JSONDecodeError: continue
                if o.get('type') == 'thinking' and (t := o.get('text')): yield {'channels': {'thought': t}}
                elif o.get('type') == 'assistant':
                    for p in (o.get('message') or {}).get('content') or []:
                        if (t := p.get('text')): yield from split.feed(t)
                elif o.get('type') == 'result': res = o
            yield from split.finish()
        if res is None: raise RuntimeError(f'cursor-agent ended without a result: {proc.stderr.read()[:400]}')
        self._step_res = self._note_usage(norm_cursor(res, self.model_id))

    def _oneshot(self, prompt, sp='', think=None, max_tokens=None):
        "Stateless one-shot text, through whichever path this chat uses."
        msg = f'{sp}\n\n{prompt}' if sp else prompt
        if self.use_sdk:
            agent = self._mk_agent()
            try: return resp_text(self._sdk_resp(agent.send(msg)))
            finally:
                try: agent.close()
                except Exception: pass
        r = self._run('json', msg)
        return resp_text(norm_cursor(json.loads(r.stdout), self.model_id))

In [ ]:
#| hide
# the command line is the safety surface, so it is worth pinning: read-only and sandboxed by default
_real, cursor_bin = cursor_bin, lambda bin=CURSOR_BIN: '/fake/cursor-agent'
try:
    c = CursorChat('cursor/grok-4.5-high')
    test_eq(c._cmd('json'), ['/fake/cursor-agent', '-p', '--output-format', 'json', '--model',
                             'grok-4.5-high', '--mode', 'ask', '--sandbox', 'enabled'])
    assert '--trust' not in c._cmd('json') and '--force' not in c._cmd('json')
    test_eq(CursorChat().model_id, grok45)                       # a default that is a real id

    # opting out is explicit, and says so on the command line
    c2 = CursorChat('cursor/composer-2.5', mode=None, sandbox=None, trust=True, workspace='/tmp/x',
                    cursor_disallowed=())
    test_eq(c2._cmd('json')[6:], ['--trust', '--workspace', '/tmp/x'])

    # `agent` is the SDK's name for the mode the CLI has no name for: its default, no flag
    test_eq(sdk_mode('agent'), 'agent')
    test_eq(cli_mode('agent'), None)
    test_eq(cli_mode('ask'), 'ask')
    test_fail(lambda: cli_mode('yolo'), contains='unknown mode')
    test_eq('--mode' in CursorChat(mode='agent', via='cli', cursor_disallowed=())._cmd('json'), False)
    # and a tool restriction the CLI cannot apply is said out loud rather than assumed
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always'); CursorChat(mode='agent', via='cli')
        assert any('no tool allowlist' in str(x.message) for x in w)

    # tools go out as tags in the prompt, because the CLI has no tool channel to put them in
    def add(a: int, b: int) -> int:
        "Add a and b."
        return a + b
    c3 = CursorChat(tools=[add], sp='Be terse.')
    c3.hist = [{'role': 'user', 'content': 'add 1 and 2'}]
    p = c3._prompt()
    assert p.startswith('Be terse.') and '<tools>' in p and '"name": "add"' in p
    assert p.rstrip().endswith('## User\nadd 1 and 2')
    test_eq(list(c3.ns), ['add'])
finally: cursor_bin = _real

## The SDK path: one agent, many turns

A `cursor-agent` turn costs about nine seconds, and only three of them are the model. The other six
are the CLI starting up: auth, session, MCP and plugin loading. rishi pays it again on every turn,
because every turn is a new process. There is no stdio server to hold open. `-p` reads stdin to EOF,
answers once and exits, and the interactive mode is a TUI.

`cursor-sdk` is the way out. One `Agent` holds the conversation across `send()` calls, so the startup
is paid once per chat instead of once per turn. It wants its own key, a user key from
[the dashboard](https://cursor.com/dashboard/api) in `$CURSOR_API_KEY`, *not* the `cursor-agent login`
session. So both paths have to stay. `via=None` takes the SDK when there is a key and the package to
use it with, and the CLI otherwise. `via='sdk'` and `via='cli'` say which outright. Neither group of
users hits a wall.

A live agent has its own memory of the conversation, so only the unsent tail goes out each turn
rather than the whole transcript. That is faster and cheaper, and it becomes a lie the moment rishi's
history changes underneath it, which is what eviction and `reconfigure` do. `_recreate_conv`, the
hook both of those already call, closes the agent. The next turn builds a fresh one and re-sends the
history as it now stands. Drift is not managed, it is made impossible.

Needs `pip install 'rishi[cursor]'`.

The CLI also reads the workspace's `AGENTS.md` and `.cursor/rules` into every turn. Asked with no
tools at all, a CLI chat in this repo still quotes them. The SDK reads neither, whatever
`setting_sources` says, and `[]` is dropped from the wire so it cannot mean *none* there. No flag
turns it off on the CLI, so a chat that wants the model rather than the workspace's agent wants
`via='sdk'`, or a `workspace` with no rules in it.


In [ ]:
#| export
@patch
def _mk_agent(self:CursorChat):
    "A live SDK agent for this chat: one login, one session, many turns."
    from cursor_sdk import Agent, AgentOptions, LocalAgentOptions, SandboxOptions
    local = LocalAgentOptions(cwd=str(self.workspace or Path.cwd()),
                              sandbox_options=None if self.sandbox is None else
                              SandboxOptions(enabled=self.sandbox not in (False, 'disabled')),
                              custom_tools=self.tool_handler.custom_tools())
    opts = AgentOptions(model=cursor_model(self.model_id, self.effort, self.fast, via='sdk'),
                        api_key=self.api_key, mode=sdk_mode(self.mode), mcp_servers={},
                        tools=self.cursor_tools, disallowed_tools=self.cursor_disallowed, local=local)
    return Agent.create(opts)

@patch(as_prop=True)
def agent(self:CursorChat):
    "The live agent, built on first use and kept until something invalidates the conversation."
    if self._agent is None: self._agent, self._sent = self._mk_agent(), 0
    return self._agent

@patch
def _tail(self:CursorChat):
    'The part of the conversation the live agent has not seen.'
    new = self.hist[self._sent:]
    skip = ('assistant', 'tool') if self.tool_channel == 'native' else ('assistant',)
    if self._sent: new = [m for m in new if m.get('role') not in skip]
    pre = self._sp() if not self._sent else ''
    return render_prompt(new, pre), len(self.hist)

@patch
def _sdk_resp(self:CursorChat, run):
    "A finished SDK `Run` -> a rishi `Resp`, through the same normalizer the CLI path uses."
    res = run.wait()
    if (st := getattr(res, 'status', 'finished')) not in ('finished', None):
        raise RuntimeError(f'cursor-sdk run {st}: {resp_text({"content": getattr(res, "result", "")}) or "no detail"}')
    u = getattr(res, 'usage', None)
    usage = {'inputTokens': getattr(u, 'input_tokens', 0), 'outputTokens': getattr(u, 'output_tokens', 0),
             'cacheReadTokens': getattr(u, 'cache_read_tokens', 0)} if u else None
    return norm_cursor({'result': getattr(res, 'result', None) or '', 'usage': usage}, self.model_id)

@patch
def _sdk_step(self:CursorChat, max_output_tokens=None):
    "One turn through the live agent: only what it has not already been told goes out."
    msg, n = self._tail()
    run = self.agent.send(msg)
    self._sent = n
    return self._note_usage(self._sdk_resp(run))

@patch
def _sdk_stream_step(self:CursorChat, max_output_tokens=None):
    "The same turn, streamed. Thinking has its own channel, and tag calls never render as prose."
    msg, n = self._tail()
    run, split = self.agent.send(msg), StreamSplit()
    self._sent = n
    for m in run.messages():
        if (t := getattr(m, 'type', None)) == 'thinking':
            if (tx := getattr(m, 'text', '')): yield {'channels': {'thought': tx}}
        elif t == 'assistant':
            for b in getattr(getattr(m, 'message', None), 'content', None) or []:
                if getattr(b, 'type', None) == 'text' and (tx := getattr(b, 'text', '')): yield from split.feed(tx)
    yield from split.finish()
    self._step_res = self._note_usage(self._sdk_resp(run))

@patch
def close(self:CursorChat):
    "Let the live agent go; the next turn builds another."
    if self._agent is not None:
        try: self._agent.close()
        except Exception: pass
        self._agent, self._sent = None, 0

In [ ]:
#| hide
# The SDK is a network client with its own key, so the adapter is tested against a stand-in: what
# matters here is what rishi sends it, and when it decides the conversation it remembers has gone stale.
import sys, types
class _FakeRun:
    def __init__(self, agent, msg): self.agent, self.msg = agent, msg
    def wait(self): return types.SimpleNamespace(status='finished', result=f'saw: {self.msg[:40]}', usage=None)
class _FakeAgent:
    live = 0
    def __init__(self, opts): self.opts, self.sent, _FakeAgent.live = opts, [], _FakeAgent.live + 1
    def send(self, msg): self.sent.append(msg); return _FakeRun(self, msg)
    def close(self): _FakeAgent.live -= 1

def _fake_chat(**kw):
    c = CursorChat(via='sdk', **kw)
    c._mk_agent = lambda: _FakeAgent(None)
    return c

_FakeAgent.live = 0
c = _fake_chat(sp='Be terse.')
test_eq(resp_text(c('hello')), 'saw: Be terse.\n\n## User\nhello')   # the briefing rides on the first message
a = c._agent
test_eq(_FakeAgent.live, 1)

# ...and only the unsent tail after that: the agent remembers the rest itself
c('and again')
test_eq(c._agent, a)                                       # same agent, no second startup
test_eq(len(a.sent), 2)
assert a.sent[1].startswith('## User\nand again') and 'Be terse.' not in a.sent[1]

# when rishi's history moves under it, the agent is dropped rather than left believing a stale past
c.reconfigure(sp='Now be florid.')
test_eq((c._agent, c._sent, _FakeAgent.live), (None, 0, 0))
c('third')
assert c._agent is not a and 'Now be florid.' in c._agent.sent[0]    # rebuilt, re-briefed, re-sent
assert '## User\nhello' in c._agent.sent[0]                          # ...with the history it still has

# `close` is idempotent and leaves nothing running
c.close(); c.close()
test_eq(_FakeAgent.live, 0)

# which path a chat takes: explicit wins, and the auto choice needs a key to pick the SDK
test_eq(CursorChat(via='cli').use_sdk, False)
test_eq(CursorChat(via='sdk').use_sdk, True)
test_fail(lambda: CursorChat(via='http'), contains="via must be 'sdk', 'cli' or None")
test_eq(sdk_available(api_key=None) if os.getenv('CURSOR_API_KEY') else sdk_available(), bool(os.getenv('CURSOR_API_KEY')))
assert sdk_available(api_key='crsr_fake') in (True, False)   # True once `cursor-sdk` is installed


# a tool result is part of the tail - the loop's answer to a call the agent made - but the agent's
# own reply never is, since it already knows what it said
c2 = _fake_chat()
c2('start'); a2 = c2._agent
c2.hist += [{'role': 'assistant', 'content': '', 'tool_calls': [ToolCall('echo', {'x': 1})]},
            {'role': 'tool', 'name': 'echo', 'content': 'echo:1'}]
c2._model_step()
assert '## Tool result (echo)\necho:1' in a2.sent[-1]
assert 'Assistant' not in a2.sent[-1]
c2.close()


# effort and speed are one pair of arguments, spelled the way each path spells them
test_eq(cursor_model(grok45), 'grok-4.5')
test_eq(cursor_model(grok45, effort='high'), 'grok-4.5[effort=high]')
test_eq(cursor_model(grok45, effort='low', fast=True), 'grok-4.5[effort=low,fast=true]')
test_eq(cursor_model(grok45, fast=False), 'grok-4.5[fast=false]')
sel = cursor_model(grok45, effort='high', fast=True, via='sdk')
test_eq((sel.id, [(p.id, p.value) for p in sel.params]), ('grok-4.5', [('effort', 'high'), ('fast', 'true')]))
_real_bin, cursor_bin = cursor_bin, lambda bin=CURSOR_BIN: '/fake/cursor-agent'
try:
    assert '--model' in (cmd := CursorChat(grok45, via='cli', effort='high')._cmd('json'))
    test_eq(cmd[cmd.index('--model') + 1], 'grok-4.5[effort=high]')
finally: cursor_bin = _real_bin

# the SDK has no `ask`, so rishi's read-only default has to become the one it does have
test_eq((sdk_mode('ask'), sdk_mode('plan'), sdk_mode('agent'), sdk_mode(None)), ('plan', 'plan', 'agent', None))
test_fail(lambda: sdk_mode('yolo'), contains='unknown mode')

# a send that fails must not mark the conversation as delivered
class _BoomAgent(_FakeAgent):
    def send(self, msg): raise RuntimeError('network blip')
c3 = _fake_chat(sp='Be terse.'); c3._mk_agent = lambda: _BoomAgent(None)
test_fail(lambda: c3('question one'), contains='network blip')
test_eq(c3._sent, 0)                                   # nothing was delivered, so nothing is marked sent
c3._agent = _FakeAgent(None)
c3('question two')
assert 'Be terse.' in c3._agent.sent[0] and '## User\nquestion one' in c3._agent.sent[0]
c3.close()


# which channel the schemas are on: Cursor runs a custom tool only in agent mode, so a read-only chat
# keeps the tags, which rishi can run itself whatever mode the harness is in
def _add(a: int, b: int) -> int:
    "Add a and b."
    return a + b
test_eq(CursorChat(tools=[_add], via='cli', mode='agent').tool_channel, 'tags')
test_eq(CursorChat(tools=[_add], via='sdk', mode='ask').tool_channel, 'tags')
test_eq(CursorChat(tools=[_add], via='sdk', mode='agent').tool_channel, 'native')
test_eq(CursorChat(via='sdk', mode='agent').tool_channel, 'tags')      # nothing to carry

# on the native channel the schemas leave the prompt, and the handler runs the call rishi's way
cn = _fake_chat(tools=[_add], mode='agent', sp='Be terse.')
assert '<tools>' not in cn._prompt() and cn._sp() == 'Be terse.'
ct = cn.tool_handler.custom_tools()
test_eq(list(ct), ['_add'])
test_eq(ct['_add'].execute({'a': 17, 'b': 25}), '42')
test_eq([m.get('role') for m in cn.hist], ['assistant', 'tool'])
test_eq((cn.hist[-1]['content'], cn._steps), ('42', 1))

# approval and the budget govern a native call exactly as they govern a tag one
cd = _fake_chat(tools=[_add], mode='agent', approve=lambda tc: False)
test_eq(cd.tool_handler.custom_tools()['_add'].execute({'a': 1, 'b': 2}), 'Denied by human operator')

# and a native call is the agent's own memory: neither half of it goes back to it
cn('start'); sent = cn._agent
cn.tool_handler.custom_tools()['_add'].execute({'a': 1, 'b': 2})
cn._model_step()
assert 'Tool result' not in sent.sent[-1]
cn.close(); cd.close()


In [ ]:
# dispatch: the prefix names the runtime, and a `cursor-` id is recognised on its own
import rishi.core, rishi.cursor
test_eq(rishi.core.get_runtime('cursor'), rishi.cursor.CursorChat)
test_eq(split_runtime('cursor/grok-4.5-high'), ('cursor', 'grok-4.5-high'))
test_eq(infer_runtime('grok-4.5-high'), 'cursor')
test_eq(infer_runtime('grok-4'), 'cursor')                # a plain grok is still somebody else's API
test_eq(type(Chat.__new__(Chat, 'cursor/grok-4.5-low')), rishi.cursor.CursorChat)

# it is hosted, whatever the local binary suggests
test_eq(rishi.cursor.CursorChat.local, False)


# a plain Cursor id is also a hosted-API name, so `Chat` needs telling; only the decorated ones infer
test_eq(infer_runtime('grok-4.5'), 'cursor')
test_eq(type(Chat.__new__(Chat, f'cursor/{rishi.cursor.grok45}')), rishi.cursor.CursorChat)

## Against the real CLI / SDK

Live cells need `cursor-agent login` for the CLI or `$CURSOR_API_KEY` for the SDK. The SDK keeps one
agent alive across turns. The CLI pays process startup on every call.

In [ ]:
# The static aliases are useful before login; `cursor_models()` below checks what this account can reach.
CURSOR_MODELS

{'cursor_default': 'default',
 'grok45': 'grok-4.5',
 'composer25': 'composer-2.5',
 'opus5': 'claude-opus-5',
 'opus48': 'claude-opus-4-8',
 'opus47': 'claude-opus-4-7',
 'opus46': 'claude-opus-4-6',
 'opus45': 'claude-opus-4-5',
 'fable5': 'claude-fable-5',
 'sonnet5': 'claude-sonnet-5',
 'sonnet46': 'claude-sonnet-4-6',
 'sonnet45': 'claude-sonnet-4-5',
 'sonnet4': 'claude-sonnet-4',
 'haiku45': 'claude-haiku-4-5',
 'gpt56_sol': 'gpt-5.6-sol',
 'gpt56_terra': 'gpt-5.6-terra',
 'gpt56_luna': 'gpt-5.6-luna',
 'gpt55': 'gpt-5.5',
 'gpt54': 'gpt-5.4',
 'gpt54_mini': 'gpt-5.4-mini',
 'gpt54_nano': 'gpt-5.4-nano',
 'gpt53_codex': 'gpt-5.3-codex',
 'gpt52': 'gpt-5.2',
 'gpt51': 'gpt-5.1',
 'gpt5_mini': 'gpt-5-mini',
 'gemini36_flash': 'gemini-3.6-flash',
 'gemini35_flash': 'gemini-3.5-flash',
 'gemini31_pro': 'gemini-3.1-pro',
 'gemini3_flash': 'gemini-3-flash',
 'gemini25_flash': 'gemini-2.5-flash',
 'kimi_k3': 'kimi-k3',
 'kimi_k27_code': 'kimi-k2.7-code',
 'glm52': 'glm-5.2'}

In [ ]:
#| eval: false
chat = CursorChat(gemini35_flash, trust=True)
r = chat('In one sentence: what is a Kalman filter?')
print(resp_text(r))
print(chat.use)

A Kalman filter is an optimal estimation algorithm that recursively estimates the true, hidden state of a dynamic system over time by combining a mathematical model of the system's behavior with a sequence of noisy, uncertain measurements.
total=13,626|in=13,584|out=42|turns=1|model=gemini-3.5-flash


In [ ]:
#| eval: false
for chunk in chat('And in one more sentence, where would I not use one?', stream=True): print(chunk, end='')

> **🧠 Thinking**
>
> **Clarifying Kalman Filter Use**
> 
> I'm focusing on concisely answering your question about when *not* to use a Kalman filter. My current thinking is to highlight scenarios where linearity and Gaussian assumptions are significantly violated, making alternative methods more suitable.
> 
> **Refining Exclusion Criteria**
> 
> I'm refining the exclusion criteria for Kalman filters. My focus is on pinpointing situations where non-linearities or non-Gaussian noise fundamentally break the standard assumptions, making other estimation techniques clearly superior.
> 
> 

You would not use a Kalman filter in systems with highly non-linear behavior and non-Gaussian noise where particle filters or deep learning models are more appropriate, or in simple applications where a basic moving average or low-pass filter is sufficient and computationally cheaper.

In [ ]:
#| eval: false
def add(a: int, b: int) -> int:
    "Add a and b."
    return a + b

tchat = CursorChat(grok45, tools=[add], trust=True)
print(resp_text(tchat('What is 17 plus 25? Use the tool.')))
print([m['content'] for m in tchat.hist if m['role'] == 'tool'])

17 plus 25 is **42**.
['42']


In [ ]:
#| eval: false
# The SDK path, and the reason for it: turn 2 skips the startup the CLI pays on every turn.
import time
sdk_chat = CursorChat(grok45, via='sdk')
for q in ['My favourite number is 7. Reply with exactly: ok', 'What is my favourite number? Digits only.']:
    s = time.time(); print(f'{resp_text(sdk_chat(q))!r}  {time.time()-s:.1f}s')
sdk_chat.close()

'ok'  2.4s
'7'  2.1s


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()